In [11]:
import os
from PIL import Image, ExifTags
# from IPython.display import display # To display images in the notebook

# --- Helper Function to get EXIF Orientation Tag ID ---
# (This handles potential variations in Pillow versions)
try:
    TAGS_REVERSE = {v: k for k, v in ExifTags.TAGS.items()}
    ORIENTATION_TAG_ID = TAGS_REVERSE.get('Orientation')
except AttributeError:
    ORIENTATION_TAG_ID = 274 # Standard EXIF Orientation Tag ID
    print("Warning: Could not fully initialize ExifTags.TAGS. Using default Orientation ID (274).")
except Exception as e:
    ORIENTATION_TAG_ID = 274
    print(f"Warning: An error occurred while initializing ExifTags: {e}. Using default Orientation ID (274).")


# --- Function to Correct Image Orientation based on EXIF ---
def correct_image_orientation(img):
    """
    Corrects the orientation of an image based on its EXIF data.
    Returns the corrected image.
    """
    if not hasattr(img, '_getexif') or ORIENTATION_TAG_ID is None:
        return img

    try:
        exif_data = img._getexif()
    except Exception: # Handle cases where _getexif might fail
        return img

    if exif_data is None:
        return img

    orientation = exif_data.get(ORIENTATION_TAG_ID)

    if orientation == 1: # Normal
        return img
    elif orientation == 2: # Mirrored horizontal
        return img.transpose(Image.FLIP_LEFT_RIGHT)
    elif orientation == 3: # Rotated 180
        return img.rotate(180, expand=True)
    elif orientation == 4: # Mirrored vertical
        return img.rotate(180, expand=True).transpose(Image.FLIP_LEFT_RIGHT)
    elif orientation == 5: # Mirrored horizontal, then rotated 270 clockwise
        return img.transpose(Image.FLIP_LEFT_RIGHT).rotate(270, expand=True)
    elif orientation == 6: # Rotated 270 clockwise / 90 counter-clockwise
        return img.rotate(270, expand=True)
    elif orientation == 7: # Mirrored horizontal, then rotated 90 clockwise
        return img.transpose(Image.FLIP_LEFT_RIGHT).rotate(90, expand=True)
    elif orientation == 8: # Rotated 90 clockwise / 270 counter-clockwise
        return img.rotate(90, expand=True)
    else:
        return img


# --- Function to Process a Single Image ---
def process_single_image(image_path, output_path, target_width, target_height,
                         fixed_rotation=0, quality=85, skip_exif=False,
                         display_processed=False):
    """
    Processes a single image: corrects orientation, applies fixed rotation, resizes, and saves.
    Returns True on success, False on failure.
    Optionally displays the processed image in the notebook.
    """
    try:
        img = Image.open(image_path)
        original_filename = os.path.basename(image_path)
        print(f"Processing {original_filename}...")

        # # 1. Correct EXIF orientation (unless skipped)
        # if not skip_exif:
        #     print(f"  Correcting EXIF orientation...")
        #     img = correct_image_orientation(img)
        # else:
        #     print(f"  Skipping EXIF orientation correction.")

        # # 2. Apply fixed rotation (if any)
        # if fixed_rotation != 0:
        #     print(f"  Applying fixed rotation of {fixed_rotation} degrees (clockwise)...")
        #     img = img.rotate(-fixed_rotation, expand=True) # Pillow rotates counter-clockwise

        # 3. Resize
        print(f"  Resizing to {target_width}x{target_height}...")
        img_resized = img.resize((target_width, target_height), Image.Resampling.LANCZOS)

        # 4. Ensure image is in RGB mode if saving as JPEG
        base, ext = os.path.splitext(output_path)
        save_img = img_resized
        if ext.lower() in ['.jpg', '.jpeg']:
            if save_img.mode in ['RGBA', 'P']:
                print(f"  Converting image mode from {save_img.mode} to RGB for JPEG saving.")
                save_img = save_img.convert('RGB')
            save_img.save(output_path, quality=quality)
        else:
            save_img.save(output_path) # For PNG, GIF etc.

        print(f"  Saved processed image to: {output_path}")

        # if display_processed:
        #     print("  Displaying processed image:")
        #     display(img_resized.resize((min(img_resized.width, 400), min(img_resized.height, 300)))) # Display smaller version

        return True

    except FileNotFoundError:
        print(f"Error: Image file not found at {image_path}")
    except IOError as e:
        print(f"Error: Could not open/read image: {image_path}. It might be corrupted or not a valid image. Details: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while processing {image_path}: {e}")
    return False

print("Helper functions defined.")

Helper functions defined.


In [12]:
# --- Configuration ---
# IMPORTANT: Replace with your actual paths!
# Use absolute paths or paths relative to where your Jupyter Notebook server is running.
# Example:
# If your notebook is in /Users/Me/Notebooks/ and images are in /Users/Me/Pictures/Input
# INPUT_DIR = "/Users/Me/Pictures/Input"
# Or, if Input is a subfolder of where the notebook is:
# INPUT_DIR = "InputImagesFolder" # Make sure this folder exists relative to your .ipynb file

INPUT_DIR = "2025.05.13 Сравнение фотофиксации/!!Kern"  # Replace with your input directory path
OUTPUT_DIR = "norm_images" # Replace with your desired output directory path

TARGET_WIDTH = 640
TARGET_HEIGHT = 640

# Additional fixed rotation in degrees (clockwise). Applied AFTER EXIF correction.
# Set to 0 for no additional fixed rotation. Common values: 0, 90, 180, 270.
FIXED_ROTATION_ANGLE = 0

# JPEG quality for output images (1-100). Ignored for non-JPEG formats like PNG.
JPEG_QUALITY = 90

# Set to True to skip EXIF-based orientation correction entirely
SKIP_EXIF_CORRECTION = False

# Set to True to display the first few processed images in the notebook (can be slow for many images)
DISPLAY_FIRST_N_IMAGES = 3 # Set to 0 or False to disable

# --- End Configuration ---

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Input Directory: {os.path.abspath(INPUT_DIR)}")
print(f"Output Directory: {os.path.abspath(OUTPUT_DIR)}")
print(f"Target Resolution: {TARGET_WIDTH}x{TARGET_HEIGHT}")
if FIXED_ROTATION_ANGLE != 0:
    print(f"Fixed Rotation: {FIXED_ROTATION_ANGLE} degrees clockwise")
if SKIP_EXIF_CORRECTION:
    print("EXIF orientation correction will be SKIPPED.")

Input Directory: C:\Users\bhunp\python312\Scripts\local_projects\search_for_not_unique_kerns\2025.05.13 Сравнение фотофиксации\!!Kern
Output Directory: C:\Users\bhunp\python312\Scripts\local_projects\search_for_not_unique_kerns\norm_images
Target Resolution: 640x640


In [13]:
if not os.path.isdir(INPUT_DIR):
    print(f"Error: Input directory '{INPUT_DIR}' not found. Please check the path in Cell 2.")
else:
    image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff', '.tif', '.webp')
    processed_count = 0
    failed_count = 0
    images_displayed = 0

    print(f"\n--- Starting Image Processing ---")

    for filename in os.listdir(INPUT_DIR):
        if filename.lower().endswith(image_extensions):
            input_path = os.path.join(INPUT_DIR, filename)
            # You can customize the output filename if needed, here it keeps the original
            output_filename = filename
            output_path = os.path.join(OUTPUT_DIR, output_filename)

            should_display = False
            if DISPLAY_FIRST_N_IMAGES and images_displayed < DISPLAY_FIRST_N_IMAGES:
                should_display = True


            success = process_single_image(
                image_path=input_path,
                output_path=output_path,
                target_width=TARGET_WIDTH,
                target_height=TARGET_HEIGHT,
                fixed_rotation=FIXED_ROTATION_ANGLE,
                quality=JPEG_QUALITY,
                skip_exif=SKIP_EXIF_CORRECTION,
                display_processed=should_display
            )

            if success:
                processed_count += 1
                if should_display:
                    images_displayed +=1
            else:
                failed_count += 1
            print("-" * 30) # Separator
        else:
            # Optionally print skipped files, can be noisy for directories with many non-images
            # print(f"Skipping non-image file or unsupported extension: {filename}")
            pass


    print(f"\n--- Processing Complete ---")
    print(f"Successfully processed: {processed_count} image(s).")
    print(f"Failed to process: {failed_count} image(s).")
    print(f"Output saved in: {os.path.abspath(OUTPUT_DIR)}")


--- Starting Image Processing ---
Processing skv.142801_керн1-5.jpg...
  Resizing to 640x640...
  Saved processed image to: norm_images\skv.142801_керн1-5.jpg
------------------------------
Processing skv.142801_керн11-15.jpg...
  Resizing to 640x640...
  Saved processed image to: norm_images\skv.142801_керн11-15.jpg
------------------------------
Processing skv.142801_керн16-20.jpg...
  Resizing to 640x640...
  Saved processed image to: norm_images\skv.142801_керн16-20.jpg
------------------------------
Processing skv.142801_керн21-25.jpg...
  Resizing to 640x640...
  Saved processed image to: norm_images\skv.142801_керн21-25.jpg
------------------------------
Processing skv.142801_керн26-30.jpg...
  Resizing to 640x640...
  Saved processed image to: norm_images\skv.142801_керн26-30.jpg
------------------------------
Processing skv.142801_керн31-35.jpg...
  Resizing to 640x640...
  Saved processed image to: norm_images\skv.142801_керн31-35.jpg
------------------------------
Processin